# Bias Detection & Fairness Analysis

This notebook implements the bias detection and fairness analysis for the NovaCred credit application dataset.
As part of the Data Governance Task Force, the goal is to identify potential discrimination in historical lending decisions
and quantify bias using established fairness metrics.

**Analyses performed:**
1. Gender Bias — Disparate Impact Ratio (four-fifths rule)
2. Proxy Discrimination — Geographic and demographic proxies for protected attributes
3. Age-Based Discrimination Patterns
4. Interaction Effects (gender × age, gender × region)
5. Fairlearn Fairness Metrics (demographic parity, equalized odds)

## 1. Data Loading and Preparation

We load the cleaned dataset produced by the Data Engineering pipeline (`01_data_quality.ipynb`).
Additional preparation includes standardizing gender labels and computing applicant age from date of birth.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load cleaned application data
df = pd.read_csv("../data/processed/credit_applications_clean.csv")
spend = pd.read_csv("../data/processed/spending_behavior_normalized.csv")

print(f"Dataset shape: {df.shape[0]} applications, {df.shape[1]} features")
df.head()

### 1.1 Gender Standardization

The Data Quality notebook identified inconsistent gender coding: both `Male`/`Female` and abbreviated `M`/`F` formats are present.
We standardize these for consistent analysis.

In [ ]:
# Inspect raw gender distribution before standardization
print("Raw gender values:")
print(df["applicant_info_gender"].value_counts(dropna=False))
print(f"\nMissing gender: {df['applicant_info_gender'].isna().sum()} records")

In [ ]:
# Standardize gender labels: M -> Male, F -> Female
df["gender"] = df["applicant_info_gender"].replace({"M": "Male", "F": "Female"})

print("Standardized gender distribution:")
print(df["gender"].value_counts(dropna=False))

# Drop rows with missing gender for bias analysis (2 records)
df_bias = df.dropna(subset=["gender"]).copy()
print(f"\nRecords for bias analysis: {len(df_bias)} (dropped {len(df) - len(df_bias)} with missing gender)")

### 1.2 Age Computation

We derive applicant age from the date of birth field, using January 2024 as the reference date
(consistent with the dataset's `processing_timestamp` values).

In [ ]:
# Parse date of birth and compute age
df_bias["date_of_birth"] = pd.to_datetime(df_bias["applicant_info_date_of_birth"], errors="coerce")
reference_date = pd.Timestamp("2024-01-01")
df_bias["age"] = ((reference_date - df_bias["date_of_birth"]).dt.days / 365.25).round(0)

print(f"Valid age values: {df_bias['age'].notna().sum()} / {len(df_bias)}")
print(f"Missing DOB: {df_bias['age'].isna().sum()} records\n")
print(df_bias["age"].describe())

### 1.3 Geographic Region Extraction

Zip codes in the dataset follow two distinct prefixes (`10xxx` and `90xxx`), suggesting two geographic regions.
We extract a region indicator for proxy discrimination analysis.

In [ ]:
# Extract region from zip code prefix
df_bias["zip_str"] = df_bias["applicant_info_zip_code"].astype(str).str.split(".").str[0]
df_bias["region"] = df_bias["zip_str"].str[:2].map({"10": "Region_10", "90": "Region_90"})

print("Region distribution:")
print(df_bias["region"].value_counts(dropna=False))

---
## 2. Gender Bias — Disparate Impact Ratio

The **Disparate Impact Ratio (DI)** measures whether a decision process disproportionately affects a protected group.
It is defined as:

$$DI = \frac{\text{Approval rate of unprivileged group}}{\text{Approval rate of privileged group}}$$

Under the **four-fifths rule**, a DI value below **0.8** indicates potential disparate impact and warrants further investigation.

In [ ]:
# Calculate approval rates by gender
approval_by_gender = df_bias.groupby("gender")["decision_loan_approved"].agg(["sum", "count", "mean"])
approval_by_gender.columns = ["approved", "total", "approval_rate"]
approval_by_gender["denied"] = approval_by_gender["total"] - approval_by_gender["approved"]

print("Approval rates by gender:")
print(approval_by_gender)
print()

In [ ]:
# Compute Disparate Impact Ratio
female_rate = approval_by_gender.loc["Female", "approval_rate"]
male_rate = approval_by_gender.loc["Male", "approval_rate"]

disparate_impact = female_rate / male_rate

print(f"Female approval rate: {female_rate:.4f} ({female_rate*100:.1f}%)")
print(f"Male approval rate:   {male_rate:.4f} ({male_rate*100:.1f}%)")
print(f"\nDisparate Impact Ratio: {disparate_impact:.4f}")
print(f"Four-fifths threshold:  0.8000")
print(f"\n{'⚠ POTENTIAL DISPARATE IMPACT DETECTED' if disparate_impact < 0.8 else '✓ No disparate impact detected'}")
print(f"The DI ratio of {disparate_impact:.4f} is {'below' if disparate_impact < 0.8 else 'above'} the 0.8 threshold.")

In [ ]:
# Statistical significance test: Chi-squared test for independence
contingency = pd.crosstab(df_bias["gender"], df_bias["decision_loan_approved"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print("Chi-squared test of independence (gender × approval):")
print(f"  Chi² statistic: {chi2:.4f}")
print(f"  p-value:        {p_value:.6f}")
print(f"  Degrees of freedom: {dof}")
print(f"\n  Result: {'Statistically significant (p < 0.05)' if p_value < 0.05 else 'Not statistically significant'}")
print(f"  The association between gender and loan approval is {'significant' if p_value < 0.05 else 'not significant'}.")

In [ ]:
# Visualization: Approval rates by gender
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of approval rates
colors = ["#e74c3c", "#3498db"]
approval_by_gender["approval_rate"].plot(kind="bar", ax=axes[0], color=colors, edgecolor="black")
axes[0].axhline(y=male_rate * 0.8, color="red", linestyle="--", linewidth=1.5, label=f"4/5 threshold ({male_rate*0.8:.2f})")
axes[0].set_title("Loan Approval Rate by Gender", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Approval Rate")
axes[0].set_xlabel("Gender")
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].tick_params(axis="x", rotation=0)

# Annotate bars
for i, (gender, row) in enumerate(approval_by_gender.iterrows()):
    axes[0].text(i, row["approval_rate"] + 0.02, f"{row['approval_rate']:.1%}", ha="center", fontweight="bold")

# Stacked bar chart of approved vs denied
approval_by_gender[["approved", "denied"]].plot(kind="bar", stacked=True, ax=axes[1],
                                                 color=["#2ecc71", "#e74c3c"], edgecolor="black")
axes[1].set_title("Approved vs Denied by Gender", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Number of Applications")
axes[1].set_xlabel("Gender")
axes[1].legend(["Approved", "Denied"])
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig("../reports/gender_approval_rates.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDisparate Impact Ratio = {disparate_impact:.4f} (threshold: 0.8)")

---
## 3. Proxy Discrimination Analysis

Proxy discrimination occurs when a non-protected attribute (e.g., zip code, spending category) is highly correlated
with a protected attribute (e.g., gender), effectively allowing indirect discrimination even when the protected
attribute is not explicitly used in decision-making.

We investigate whether **geographic region** (derived from zip code) serves as a proxy for gender.

### 3.1 Zip Code Region as a Gender Proxy

If zip code regions strongly correlate with gender, then using zip code in a lending model
could introduce indirect gender discrimination — even without directly considering gender.

In [ ]:
# Cross-tabulation: gender × region
df_proxy = df_bias.dropna(subset=["region"]).copy()

gender_region = pd.crosstab(df_proxy["gender"], df_proxy["region"], margins=True)
print("Gender × Region Cross-Tabulation:")
print(gender_region)

# Chi-squared test for association
ct = pd.crosstab(df_proxy["gender"], df_proxy["region"])
chi2_proxy, p_proxy, dof_proxy, _ = stats.chi2_contingency(ct)
print(f"\nChi-squared test (gender × region):")
print(f"  Chi² = {chi2_proxy:.2f}, p-value = {p_proxy:.2e}")
print(f"  {'Strong association detected' if p_proxy < 0.001 else 'No significant association'}")

# Cramér's V for effect size
n = ct.sum().sum()
cramers_v = np.sqrt(chi2_proxy / (n * (min(ct.shape) - 1)))
print(f"  Cramér's V = {cramers_v:.4f} ({'Strong' if cramers_v > 0.3 else 'Moderate' if cramers_v > 0.1 else 'Weak'} association)")

In [ ]:
# Visualization: Gender distribution across regions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gender composition per region
gender_region_pct = pd.crosstab(df_proxy["region"], df_proxy["gender"], normalize="index")
gender_region_pct.plot(kind="bar", stacked=True, ax=axes[0], color=["#e74c3c", "#3498db"], edgecolor="black")
axes[0].set_title("Gender Composition by Region", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Proportion")
axes[0].set_xlabel("Region")
axes[0].legend(title="Gender")
axes[0].tick_params(axis="x", rotation=0)
axes[0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

# Approval rates by region
approval_by_region = df_proxy.groupby("region")["decision_loan_approved"].mean()
approval_by_region.plot(kind="bar", ax=axes[1], color=["#f39c12", "#8e44ad"], edgecolor="black")
axes[1].set_title("Approval Rate by Region", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Approval Rate")
axes[1].set_xlabel("Region")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=0)

for i, (region, rate) in enumerate(approval_by_region.items()):
    axes[1].text(i, rate + 0.02, f"{rate:.1%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/proxy_discrimination_region.png", dpi=150, bbox_inches="tight")
plt.show()

print("\\nKey finding: If Region is highly correlated with Gender, then using zip code")
print("in lending decisions can serve as a proxy for gender discrimination.")

### 3.2 Spending Categories as Potential Proxies

Certain spending categories may correlate with gender and could introduce bias if used as model features.
We analyze whether spending patterns differ systematically between genders.

In [ ]:
# Merge spending data with gender information
spend_gender = spend.merge(df_bias[["_id", "gender"]], on="_id", how="inner")

# Category prevalence by gender
cat_gender = pd.crosstab(spend_gender["category"], spend_gender["gender"], normalize="columns")
print("Spending category distribution by gender (column-normalized):")
print(cat_gender.round(3))

# Visualize spending category differences
fig, ax = plt.subplots(figsize=(12, 6))
cat_gender.plot(kind="bar", ax=ax, color=["#e74c3c", "#3498db"], edgecolor="black")
ax.set_title("Spending Category Distribution by Gender", fontsize=14, fontweight="bold")
ax.set_ylabel("Proportion of Spending Records")
ax.set_xlabel("Category")
ax.legend(title="Gender")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("../reports/spending_by_gender.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Age-Based Discrimination Patterns

Age is a protected characteristic under many anti-discrimination frameworks.
We investigate whether older or younger applicants face systematically different approval outcomes.

In [ ]:
# Age-based analysis (only records with valid DOB)
df_age = df_bias.dropna(subset=["age"]).copy()

# Create age groups for analysis
df_age["age_group"] = pd.cut(df_age["age"], bins=[20, 30, 40, 50, 60, 70],
                              labels=["21-30", "31-40", "41-50", "51-60", "61-70"])

# Approval rate by age group
approval_by_age = df_age.groupby("age_group", observed=True)["decision_loan_approved"].agg(["mean", "count"])
approval_by_age.columns = ["approval_rate", "count"]
print("Approval rates by age group:")
print(approval_by_age)

# Statistical test: correlation between age and approval
point_biserial, pb_pvalue = stats.pointbiserialr(
    df_age["decision_loan_approved"].astype(int), df_age["age"]
)
print(f"\nPoint-biserial correlation (age × approval): r = {point_biserial:.4f}, p = {pb_pvalue:.4f}")
print(f"{'Significant age effect detected' if pb_pvalue < 0.05 else 'No significant age effect'}")

In [ ]:
# Visualization: Approval rate by age group
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of approval rates by age group
approval_by_age["approval_rate"].plot(kind="bar", ax=axes[0], color="#2ecc71", edgecolor="black")
axes[0].set_title("Approval Rate by Age Group", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Approval Rate")
axes[0].set_xlabel("Age Group")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)

for i, (group, row) in enumerate(approval_by_age.iterrows()):
    axes[0].text(i, row["approval_rate"] + 0.02, f"{row['approval_rate']:.1%}\n(n={int(row['count'])})",
                 ha="center", fontsize=9)

# Age distribution of approved vs denied
approved_ages = df_age[df_age["decision_loan_approved"] == True]["age"]
denied_ages = df_age[df_age["decision_loan_approved"] == False]["age"]

axes[1].hist([approved_ages, denied_ages], bins=15, label=["Approved", "Denied"],
             color=["#2ecc71", "#e74c3c"], edgecolor="black", alpha=0.8)
axes[1].set_title("Age Distribution: Approved vs Denied", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/age_discrimination.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Interaction Effects

Bias may not be visible when examining a single attribute in isolation. Interaction effects reveal
how combinations of protected and non-protected attributes compound disadvantage.
We examine **gender × age** and **gender × region** interactions.

### 5.1 Gender × Age Interaction

Do younger women face even lower approval rates compared to younger men? 
This analysis reveals whether age and gender compound disadvantage.

In [ ]:
# Gender × Age group interaction
df_interact = df_bias.dropna(subset=["age"]).copy()
df_interact["age_group"] = pd.cut(df_interact["age"], bins=[20, 35, 50, 70],
                                   labels=["Young (21-35)", "Middle (36-50)", "Senior (51-70)"])

interaction = df_interact.groupby(["gender", "age_group"], observed=True)["decision_loan_approved"].agg(["mean", "count"])
interaction.columns = ["approval_rate", "count"]
print("Approval rate by Gender × Age Group:")
print(interaction)
print()

# Compute DI for each age group
for age_grp in df_interact["age_group"].dropna().unique():
    subset = df_interact[df_interact["age_group"] == age_grp]
    rates = subset.groupby("gender")["decision_loan_approved"].mean()
    if "Female" in rates.index and "Male" in rates.index:
        di = rates["Female"] / rates["Male"]
        print(f"  DI ratio for {age_grp}: {di:.4f} {'⚠ below 0.8' if di < 0.8 else '✓ above 0.8'}")

In [ ]:
# Visualization: Interaction heatmap
interaction_pivot = interaction["approval_rate"].unstack(level="age_group")

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(interaction_pivot, annot=True, fmt=".1%", cmap="RdYlGn", vmin=0.3, vmax=0.9,
            linewidths=1, ax=ax, cbar_kws={"label": "Approval Rate"})
ax.set_title("Approval Rate: Gender × Age Group Interaction", fontsize=14, fontweight="bold")
ax.set_ylabel("Gender")
ax.set_xlabel("Age Group")

plt.tight_layout()
plt.savefig("../reports/interaction_gender_age.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Gender × Region Interaction

Since zip code is a strong proxy for gender, we examine whether the gender gap persists
within each region, or if the overall disparity is driven entirely by regional differences.

In [ ]:
# Gender × Region interaction
region_interaction = df_proxy.groupby(["gender", "region"])["decision_loan_approved"].agg(["mean", "count"])
region_interaction.columns = ["approval_rate", "count"]
print("Approval rate by Gender × Region:")
print(region_interaction)
print()

# DI within each region
for reg in df_proxy["region"].dropna().unique():
    subset = df_proxy[df_proxy["region"] == reg]
    rates = subset.groupby("gender")["decision_loan_approved"].mean()
    if "Female" in rates.index and "Male" in rates.index and rates["Male"] > 0:
        di = rates["Female"] / rates["Male"]
        print(f"  DI ratio in {reg}: {di:.4f} {'⚠ below 0.8' if di < 0.8 else '✓ above 0.8'} (F: n={len(subset[subset['gender']=='Female'])}, M: n={len(subset[subset['gender']=='Male'])})")

---
## 6. Fairlearn Fairness Metrics

We use the [Fairlearn](https://fairlearn.org/) library to compute established fairness metrics,
providing a standardized assessment of algorithmic fairness.

- **Demographic Parity Difference**: measures difference in positive outcome rates between groups
- **Equalized Odds Difference**: measures difference in error rates between groups

In [ ]:
from fairlearn.metrics import (
    demographic_parity_difference,
    demographic_parity_ratio,
    MetricFrame,
    selection_rate
)

# Prepare data for fairlearn
y_true = df_bias["decision_loan_approved"].astype(int)
sensitive_gender = df_bias["gender"]

# Demographic parity metrics (treating loan decisions as model predictions)
dp_diff = demographic_parity_difference(y_true, y_true, sensitive_features=sensitive_gender)
dp_ratio = demographic_parity_ratio(y_true, y_true, sensitive_features=sensitive_gender)

print("Fairlearn Demographic Parity Metrics (Gender):")
print(f"  Demographic Parity Difference: {dp_diff:.4f}")
print(f"  Demographic Parity Ratio:      {dp_ratio:.4f}")
print(f"  (A ratio < 0.8 indicates potential unfairness)")
print()

# MetricFrame for detailed per-group analysis
mf = MetricFrame(
    metrics={"selection_rate": selection_rate, "count": lambda y_true, y_pred: len(y_true)},
    y_true=y_true,
    y_pred=y_true,
    sensitive_features=sensitive_gender
)
print("Per-group metrics:")
print(mf.by_group)
print(f"\nOverall selection rate: {mf.overall['selection_rate']:.4f}")

In [ ]:
# Fairlearn metrics with region as sensitive feature
sensitive_region = df_proxy["region"]
y_region = df_proxy["decision_loan_approved"].astype(int)

dp_diff_region = demographic_parity_difference(y_region, y_region, sensitive_features=sensitive_region)
dp_ratio_region = demographic_parity_ratio(y_region, y_region, sensitive_features=sensitive_region)

print("Fairlearn Demographic Parity Metrics (Region as proxy):")
print(f"  Demographic Parity Difference: {dp_diff_region:.4f}")
print(f"  Demographic Parity Ratio:      {dp_ratio_region:.4f}")

# Visualization: Fairlearn summary
fig, ax = plt.subplots(figsize=(8, 5))
metrics_summary = pd.DataFrame({
    "Metric": ["DI Ratio\n(Gender)", "DP Ratio\n(Gender)", "DP Ratio\n(Region)"],
    "Value": [disparate_impact, dp_ratio, dp_ratio_region]
})
bars = ax.bar(metrics_summary["Metric"], metrics_summary["Value"],
              color=["#e74c3c" if v < 0.8 else "#2ecc71" for v in metrics_summary["Value"]],
              edgecolor="black")
ax.axhline(y=0.8, color="red", linestyle="--", linewidth=2, label="Fairness threshold (0.8)")
ax.set_title("Fairness Metrics Summary", fontsize=14, fontweight="bold")
ax.set_ylabel("Ratio")
ax.set_ylim(0, 1.1)
ax.legend()

for bar, val in zip(bars, metrics_summary["Value"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}",
            ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/fairlearn_summary.png", dpi=150, bbox_inches="tight")
plt.show()